# Nota → icra modeli v1 · eğitim · tek hücre

**Yapacağın üç şey:**
1. Üst menü: **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU → Kaydet**.
2. Aşağıdaki hücrenin solundaki **▶** düğmesine bas.
3. Açılan pencerede Google hesabını seçip **Drive'a izin ver**. **veri:** satırını görünce kalkabilirsin.

Gerisi kendiliğinden: veri, eğitim, değerlendirme ve ölçüt pasajları için modelin çaldığı E dosyaları. Bitince
çalışma zamanı kapanır.

- Bağlantı koparsa `DEVAM = 'son'` yapıp yeniden ▶: eğitim son kayıt noktasından sürer.
- Sonuç Drive'da: `ses_nota/icra/runs/<tarih>-<AD>/` (rapor.md, degerlendirme.json, E/).

**İÇ TEST.** Veri ve model yalnız Drive'da kalır; paylaşılmaz.


In [ ]:
# Nota -> icra modeli v1 -- TEK HÜCRE. A100.
# Yazılı notadan icra: nota başına zamanlama, vuruş gücü, bağlı, perde sapması, titreşim, eğim, glissando, atak/bitiş,
# süslemeler. İki yönlü nota kodlayıcısı + önceki notaların örneklenmiş çıktılarına bakan nedensel çözücü; her hedef
# kutulanmış kategorik dağılım (sıcaklıkla örneklenir).
# Veri: ses_nota/icra/<PAKET>/ · kod: ses_nota/icra/kod.zip · çıktı: ses_nota/icra/runs/<tarih>-<AD>/
AD = 'icra-v1'        # koşu adı: runs/<tarih>-<AD>/ (aynı gün ikinci koşu -2, -3 ...; eski koşunun üstüne yazılmaz)
ADIM = 4000           # eğitim adımı (parti 64 kesit x 128 nota, tam veride ~50 tur; doğrulamada en iyi nokta saklanır)
DEVAM = None          # oturum kesilirse: 'son' (bu AD'li en son koşunun son kayıt noktasından sürer)
KAPAT = True          # bitince Drive'ı kaydedip çalışma zamanını kapat (kota harcamasın)
A100_SART = True      # A100 yoksa eğitime başlamadan dur · False: yine de koş
PAKET = 'icra-v1b-paket'
ORNEK = 3             # her ölçüt pasajı için kaç örnek (tohum 0, 1, 2)
SICAKLIK = 1.0        # örnekleme sıcaklığı
# ----------------------------------------------------------------------------------------------------
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU yok'
print('GPU:', GPU)
if A100_SART and 'A100' not in GPU:
    raise SystemExit(f'A100 değil ({GPU}): Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU seç, yeniden '
                     'çalıştır. A100 olmadan koşmak istersen A100_SART = False yap.')
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil, zipfile
DRIVE = '/content/drive/MyDrive/ses_nota'
IK = f'{DRIVE}/icra'
KOD = '/content/icra-kod'
shutil.rmtree(KOD, ignore_errors=True)
zipfile.ZipFile(f'{IK}/kod.zip').extractall(KOD)
sys.path.insert(0, KOD)
import icra_colab as ic
_kapat = KAPAT
A = None
try:
    A = ic.ayarlar(AD, ADIM, DEVAM, IK, GPU, KOD, PARTI=64, ORNEK=ORNEK, SICAKLIK=SICAKLIK, PAKET=PAKET)
    ic.veri(A)            # Drive'dan /content'e, shard ve boyut denetimiyle
    ic.hazirlik(A)        # koşu klasörü; DEVAM ise son kayıt noktası
    ic.egit(A)            # ADIM'a kadar; 10 dk'da bir ve doğrulamada en iyi olunca kayıt noktası Drive'a
    ic.degerlendir(A)     # kafa başına NLL, örnek/ortalama/yazılı-zaman tabanlarıyla dağılımlar, kurulum hatası
    ic.E(A)               # ölçüt pasajları için E/<id>.json (+ ornek-1, ornek-2), şema denetimiyle
except KeyboardInterrupt:  # elle durduruldu: çalışma zamanı açık kalsın
    _kapat = False
    raise
finally:
    if A is not None:
        ic.ozet(A)
    print('BİTTİ.')
    if _kapat:
        print('Drive kaydediliyor, çalışma zamanı kapatılıyor (KAPAT = True)...')
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()
    else:
        print('Çalışma zamanı → Çalışma zamanının bağlantısını kes ve sil (A100 boşta da kota harcar).')
